# 🗳️ Thai Election OCR 2026 — Typhoon Local + Typhoon OCR
## Score Target: > 0.50 (Best known: 0.5247)

---

## 🏗️ Full Pipeline

```
[Cell 1]  🖥️  ตรวจ Environment: RunPod หรือ Colab + GPU Info
[Cell 2]  📦  Install Dependencies
[Cell 3]  🚀  Start vLLM Server: typhoon-ai/typhoon-ocr-3b
[Cell 4]  📂  Kaggle Setup + Unzip Dataset
[Cell 5]  🔧  Helper Functions (Thai normalize, JSON parser, Cache)
[Cell 6]  🖼️  HTML Table Cleaner (ทำความสะอาด OCR output)
[Cell 7]  🔍  Table Filter (ตัดหน้าปก/ลายเซ็นออก ด้วย OpenCV)
[Cell 8]  🧠  OCR Functions: typhoon_ocr() + extract_votes()
[Cell 9]  ⚙️  Ingest Template CSV → doc_to_images map
[Cell 10] 🔄  Main Pipeline (Process 300 docs)
[Cell 11] ✅  Final Check + Export submission.csv
[Cell 12] 🐛  Debug: ทดสอบ 1 ภาพ
```

---

## 🖥️ Environment Support

| Platform | GPU | VRAM | Status |
|---|---|---|---|
| **RunPod** | A100 / H100 | 40-80GB | ✅ แนะนำ |
| **RunPod** | RTX 3090/4090 | 24GB | ✅ ดี |
| Colab Pro+ | A100 | 40GB | ✅ ดี |
| Colab Pro | L4 | 24GB | ✅ พอได้ |

---

## 🤖 Algorithm & Technique

| ขั้นตอน | วิธี | เหตุผล |
|---|---|---|
| **Table Filter** | OpenCV morphology (H+V lines) | ตัดหน้าปก/ลายเซ็นออก ลด noise |
| **HTML Cleaner** | BeautifulSoup parse | OCR คืน HTML → extract ข้อมูลตาราง |
| **OCR** | Typhoon OCR 3B (vLLM) | Fine-tuned Thai doc, structure prompt |
| **Extract** | Typhoon LLM Thai prompt | 2-stage: VLM→LLM ลด hallucination |
| **Fuzzy Match** | RapidFuzz token_sort_ratio ≥78 | รับมือ OCR อ่านชื่อพรรคผิดนิดหน่อย |
| **Zero Fix** | Reprompt focus เฉพาะพรรค | แก้พรรคที่ได้ 0 |
| **Cache** | Disk cache per page | ไม่ต้อง re-run ถ้า kernel restart |


In [1]:
from google.colab import files
files.upload()

KeyboardInterrupt: 

In [3]:
!unzip /content/super-ai-engineer-season-6-ocr-2569.zip

Archive:  /content/super-ai-engineer-season-6-ocr-2569.zip
  inflating: final_data/images/constituency_10_1.png  
  inflating: final_data/images/constituency_10_10.png  
  inflating: final_data/images/constituency_10_10_page2.png  
  inflating: final_data/images/constituency_10_11.png  
  inflating: final_data/images/constituency_10_11_page2.png  
  inflating: final_data/images/constituency_10_11_page3.png  
  inflating: final_data/images/constituency_10_12.png  
  inflating: final_data/images/constituency_10_12_page2.png  
  inflating: final_data/images/constituency_10_12_page3.png  
  inflating: final_data/images/constituency_10_13.png  
  inflating: final_data/images/constituency_10_13_page2.png  
  inflating: final_data/images/constituency_10_14.png  
  inflating: final_data/images/constituency_10_14_page2.png  
  inflating: final_data/images/constituency_10_14_page3.png  
  inflating: final_data/images/constituency_10_16.png  
  inflating: final_data/images/constituency_10_16_page

In [4]:
# ════════════════════════════════════════════════════════════════════
#  CELL 1  ตรวจ Environment: RunPod หรือ Colab + GPU Info
#
#  Critical Thinking:
#  - RunPod: รัน Jupyter บน pod ที่เช่า GPU → path ต่างจาก Colab
#  - A100/H100: ต้องใช้ --disable-custom-all-reduce (NCCL fix)
#  - enforce-eager: ปิด CUDA graph → stable บน single-GPU environment
# ════════════════════════════════════════════════════════════════════
import subprocess, os, sys, torch

# ── ตรวจว่า RunPod หรือ Colab ──────────────────────────────────────
IS_RUNPOD = os.path.exists('/workspace') or 'RUNPOD_POD_ID' in os.environ
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_RUNPOD:
    ENV_NAME  = 'RunPod'
    BASE_DIR  = '/workspace'
elif IS_COLAB:
    ENV_NAME  = 'Colab'
    BASE_DIR  = '/content'
else:
    ENV_NAME  = 'Local'
    BASE_DIR  = os.path.expanduser('~')

print(f'Environment : {ENV_NAME}')
print(f'Base dir    : {BASE_DIR}')

# ── ตั้ง Paths ────────────────────────────────────────────────────
DATA_DIR      = f'{BASE_DIR}/final_data/images'
TEMPLATE_PATH = f'{BASE_DIR}/final_data/submission_template_v3.csv'
OUTPUT_PATH   = f'{BASE_DIR}/submission.csv'
CACHE_DIR     = f'{BASE_DIR}/ocr_cache'
LOG_PATH      = f'{BASE_DIR}/vllm_server.log'

os.makedirs(CACHE_DIR, exist_ok=True)

# ── GPU Info ──────────────────────────────────────────────────────
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('='*55)
print('  GPU Information')
print('='*55)
print(r.stdout.strip())

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU — ต้องการ GPU สำหรับรัน vLLM')

vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name  = torch.cuda.get_device_name(0)
gpu_upper = gpu_name.upper()

print(f'\n  GPU : {gpu_name}')
print(f'  VRAM: {vram_gb:.1f} GB')

# ── ตั้งค่า vLLM params ตาม VRAM ───────────────────────────────
if vram_gb >= 70:    MAX_MODEL_LEN, GPU_TIER = 32768, 'H100'
elif vram_gb >= 35:  MAX_MODEL_LEN, GPU_TIER = 16384, 'A100'
elif vram_gb >= 20:  MAX_MODEL_LEN, GPU_TIER = 16000, 'L4/3090/4090'
else:                MAX_MODEL_LEN, GPU_TIER = 8192,  'T4'

# A100/H100: ต้องปิด NCCL custom all-reduce
NEED_DISABLE_CUSTOM_REDUCE = any(x in gpu_upper for x in ['A100','H100','A800','H800'])

print(f'  Tier                      : {GPU_TIER}')
print(f'  max-model-len             : {MAX_MODEL_LEN}')
print(f'  disable-custom-all-reduce : {NEED_DISABLE_CUSTOM_REDUCE}')
print(f'\n✅ Config ready')


Environment : Colab
Base dir    : /content


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 2  Install Dependencies
#
#  RunPod Note: ใช้ PyTorch image (runpod/pytorch:2.x.x-py3.x-cudaXX)
#  → torch และ cuda ติดตั้งมาแล้ว ไม่ต้องติดตั้งใหม่
# ════════════════════════════════════════════════════════════════════

# ลบ opentelemetry เก่าออก (ป้องกัน version conflict กับ vLLM)
!pip uninstall -q -y opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http 2>/dev/null || true

# vLLM (รองรับ Typhoon OCR 3B ซึ่ง based on Qwen2.5-VL)
!pip install -q vllm

# typhoon-ocr package
!pip install -q typhoon-ocr

# OpenCV สำหรับ Table Filter
!pip install -q opencv-python-headless

# BeautifulSoup สำหรับ HTML Table Cleaner
!pip install -q beautifulsoup4 lxml

# Utilities
!pip install -q pandas tqdm pillow rapidfuzz openai kaggle

import subprocess
r = subprocess.run(['pip', 'show', 'vllm'], capture_output=True, text=True)
ver = [l for l in r.stdout.split('\n') if 'Version' in l]
print('✅ Install done')
if ver: print(f'  {ver[0]}')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 3  Start vLLM Server: typhoon-ai/typhoon-ocr-3b
#
#  โมเดล: Qwen2.5-VL-3B-Instruct fine-tuned สำหรับ OCR เอกสารไทย
#  ขนาด: ~6GB disk | ~7GB VRAM (bfloat16)
#  Download: ครั้งแรก 5-10 นาที (HuggingFace → pod storage)
#
#  Key Flags:
#  --enforce-eager           → stable, ปิด CUDA graph
#  --disable-custom-all-reduce → แก้ NCCL crash บน A100/H100
#  --dtype bfloat16          → A100/H100 native format
# ════════════════════════════════════════════════════════════════════
import subprocess, time
import requests as req_lib

TYPHOON_MODEL = 'typhoon-ai/typhoon-ocr-3b'
VLLM_PORT     = 8000
VLLM_HOST     = f'http://localhost:{VLLM_PORT}/v1'
GPU_UTIL      = 0.88
USE_CACHE     = True
MAX_RETRIES   = 3


def build_vllm_cmd():
    cmd = [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model',                  TYPHOON_MODEL,
        '--served-model-name',      'typhoon-ocr',
        '--max-model-len',          str(MAX_MODEL_LEN),
        '--gpu-memory-utilization', str(GPU_UTIL),
        '--trust-remote-code',
        '--dtype',                  'bfloat16',
        '--port',                   str(VLLM_PORT),
        '--limit-mm-per-prompt',    'image=5',
        '--enforce-eager',
        '--disable-log-stats',
        '--max-num-seqs',           '4',
    ]
    if NEED_DISABLE_CUSTOM_REDUCE:
        cmd.append('--disable-custom-all-reduce')
        print(f'  ⚡ {GPU_TIER}: --disable-custom-all-reduce enabled')
    return cmd


def start_vllm():
    cmd = build_vllm_cmd()
    print(f'▶ Starting vLLM on {ENV_NAME}')
    print(f'  Model : {TYPHOON_MODEL}')
    print(f'  GPU   : {GPU_TIER} ({vram_gb:.0f}GB)')
    print(f'  Port  : {VLLM_PORT}')
    print(f'  Download ~6GB (ครั้งแรกใช้เวลา 5-10 นาที)...')
    log_file = open(LOG_PATH, 'w')
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    return proc, log_file


def wait_for_server(proc, timeout=900):
    start = time.time()
    print('  รอ server โหลดโมเดล', end='')
    while time.time() - start < timeout:
        if proc.poll() is not None:
            print('\n❌ vLLM crashed!')
            try:
                print(open(LOG_PATH).read()[-3000:])
            except: pass
            return False
        try:
            r = req_lib.get(f'{VLLM_HOST}/models', timeout=5)
            if r.status_code == 200:
                models   = r.json().get('data', [])
                model_id = models[0]['id'] if models else '?'
                print(f'\n✅ Ready! Model: {model_id}')
                return True
        except: pass
        print('.', end='', flush=True)
        time.sleep(10)
    print(f'\n❌ Timeout — ดู log: {LOG_PATH}')
    return False


vllm_proc, vllm_log = start_vllm()
vllm_ready          = wait_for_server(vllm_proc, timeout=900)

if vllm_ready:
    from openai import OpenAI
    vllm_client = OpenAI(api_key='no-key', base_url=VLLM_HOST)
    print(f'\n🚀 vLLM client ready @ {VLLM_HOST}')
else:
    vllm_client = None
    raise RuntimeError(f'vLLM ไม่พร้อม — ดู log: {LOG_PATH}')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 3b  Debug: ดู vLLM Log (รันถ้า server ไม่ขึ้น)
# ════════════════════════════════════════════════════════════════════
try:
    print(open(LOG_PATH).read()[-5000:])
except FileNotFoundError:
    print('ยังไม่มี log — รัน Cell 3 ก่อน')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 5  Helper Functions
#
#  Techniques:
#  - Thai digit normalization: ๐-๙ → 0-9
#  - Robust JSON parser: รองรับ ```json blocks + partial JSON
#  - img_to_b64: resize max 1800px ลด token ใช้
#  - Disk cache: ไม่ต้อง re-run ถ้า kernel restart
#  - Fuzzy match: token_sort_ratio ≥78 จับชื่อพรรคที่ OCR อ่านผิดนิดหน่อย
# ════════════════════════════════════════════════════════════════════
import io, re, json, base64, time
from pathlib     import Path
from typing      import Dict, List, Optional, Any

import pandas as pd
from tqdm      import tqdm
from PIL       import Image
from rapidfuzz import fuzz, process as rfprocess


# ── Thai digit normalizer ─────────────────────────────────────────
_THAI = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

def normalize_vote(raw: Any) -> str:
    """แปลงค่าใดๆ → string ตัวเลข Arabic, คืน '0' ถ้าไม่มี"""
    if raw is None: return '0'
    text   = str(raw).translate(_THAI).split('(')[0]
    # ปราบอักษรอังกฤษที่หน้าตาเหมือนตัวเลข
    text   = text.replace('l','1').replace('I','1').replace('O','0').replace('o','0')
    text   = text.replace('s','5').replace('S','5').replace('B','8')
    # ปราบ ด.เด็ก (ไทย ด ที่ OCR อ่านแทน 1)
    text   = re.sub(r'(?<=[0-9])ด', '1', text)
    text   = re.sub(r'ด(?=[0-9])', '1', text)
    digits = re.sub(r'[^\d]', '', text)
    return digits.lstrip('0') or '0' if digits else '0'


def parse_json(text: str) -> Optional[Dict]:
    """Robust JSON parser: รองรับ ```json blocks และ partial JSON"""
    if not text: return None
    text = re.sub(r'```(?:json)?', '', text).strip().strip('`').strip()
    try: return json.loads(text)
    except:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except: pass
    return None


def parse_typhoon_output(text: str) -> str:
    """Typhoon คืน JSON {\"natural_text\": \"...\"} → ดึง text ออกมา"""
    if not text: return ''
    data = parse_json(text)
    if data:
        for key in ['natural_text', 'text', 'table', 'html', 'markdown', 'content']:
            if key in data and data[key]: return str(data[key])
    return text


def load_cache(key: str) -> Optional[str]:
    if not USE_CACHE: return None
    p = Path(CACHE_DIR) / f'{key}.txt'
    return p.read_text('utf-8') if p.exists() else None

def save_cache(key: str, text: str):
    if USE_CACHE: (Path(CACHE_DIR) / f'{key}.txt').write_text(text, 'utf-8')


def img_to_b64(path: str, max_dim: int = 1800) -> str:
    """Resize + แปลงเป็น base64 PNG (ลด token)"""
    img  = Image.open(path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        img   = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def find_party_vote(party: str, data: Dict, threshold: int = 78) -> str:
    """Exact match → fuzzy match (token_sort_ratio ≥78)"""
    if not data: return '0'
    if party in data: return normalize_vote(data[party])
    for k, v in data.items():
        if k.strip() == party.strip(): return normalize_vote(v)
    result = rfprocess.extractOne(party, list(data.keys()), scorer=fuzz.token_sort_ratio)
    if result and result[1] >= threshold:
        return normalize_vote(data[result[0]])
    return '0'


print('✅ Helper functions ready')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 6  HTML Table Cleaner
#
#  ทำไมต้องมี HTML Cleaner?
#  - Typhoon OCR mode='structure' คืน HTML table
#  - HTML อาจมี tag ขยะ, entity, nbsp ปนมา
#  - Cleaner แปลง HTML → plain text ที่ LLM อ่านง่ายขึ้น
#  - ลด hallucination ในขั้นตอน extract_votes
# ════════════════════════════════════════════════════════════════════
import re
from typing import Optional

try:
    from bs4 import BeautifulSoup
    BS4_OK = True
except ImportError:
    BS4_OK = False
    print('⚠️ beautifulsoup4 ไม่มี → ใช้ regex fallback')


def clean_html_table(html_text: str) -> str:
    """
    แปลง HTML table → plain text สะอาด
    Input  : '<table><tr><td>เพื่อไทย</td><td>14,813</td></tr></table>'
    Output : 'เพื่อไทย 14813'
    """
    if not html_text: return ''

    if BS4_OK and '<table' in html_text.lower():
        soup  = BeautifulSoup(html_text, 'lxml')
        rows  = []
        for tr in soup.find_all('tr'):
            cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
            if cells:
                rows.append('\t'.join(cells))
        if rows:
            result = '\n'.join(rows)
        else:
            result = soup.get_text(separator=' ', strip=True)
    else:
        # Regex fallback
        result = re.sub(r'<br\s*/?>', '\n', html_text, flags=re.IGNORECASE)
        result = re.sub(r'</tr>', '\n', result, flags=re.IGNORECASE)
        result = re.sub(r'</td>|</th>', '\t', result, flags=re.IGNORECASE)
        result = re.sub(r'<[^>]+>', '', result)

    # Clean up
    result = result.replace('&nbsp;', ' ').replace('&amp;', '&')
    result = result.replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
    result = re.sub(r'[ \t]+', ' ', result)      # ลด whitespace
    result = re.sub(r'\n{3,}', '\n\n', result)   # ลด blank lines
    result = result.strip()

    return result


def clean_ocr_output(ocr_text: str) -> str:
    """
    Clean pipeline OCR output:
    1. ถ้ามี HTML table → extract ด้วย BeautifulSoup
    2. แปลง Thai digits → Arabic
    3. ลบ noise characters
    """
    if not ocr_text: return ''

    # แยก pages ออก แล้วทำ clean แต่ละหน้า
    pages  = ocr_text.split('---page---')
    cleaned = []

    for page in pages:
        page = page.strip()
        if not page: continue

        # ถ้ามี HTML table
        if '<table' in page.lower() or '<tr' in page.lower():
            page = clean_html_table(page)

        # แปลง Thai digits
        page = page.translate(_THAI)

        # ลบ noise: | (pipe ที่ OCR มองว่าเป็น table border)
        page = re.sub(r'\s*\|\s*', ' ', page)

        cleaned.append(page)

    return '\n\n'.join(cleaned)


# ── ทดสอบ HTML Cleaner ────────────────────────────────────────────
test_html = """
<table>
  <tr><th>ลำดับ</th><th>พรรค</th><th>คะแนน</th></tr>
  <tr><td>1</td><td>เพื่อไทย</td><td>14,813</td></tr>
  <tr><td>2</td><td>ประชาธิปัตย์</td><td>8,204</td></tr>
  <tr><td>3</td><td>ภูมิใจไทย</td><td>5,102</td></tr>
</table>
"""
cleaned = clean_html_table(test_html)
print('=== HTML Cleaner Test ===')
print(cleaned)
print('\n✅ HTML Cleaner ready')
print(f'  BeautifulSoup: {"✅" if BS4_OK else "❌ ใช้ regex fallback"}')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 7  Table Filter (ตัดหน้าปก/ลายเซ็นออก)
#
#  เทคนิค: OpenCV Morphological Lines
#  1. แปลงภาพเป็น binary (inverse threshold)
#  2. สกัดเส้นนอน (horizontal kernel) + เส้นตั้ง (vertical kernel)
#  3. ถ้าพบ grid ที่ใหญ่กว่า 10% ของภาพ → นับว่าเป็น 'หน้าตาราง'
#
#  ผลลัพธ์: ลด noise จากหน้าปก/ลายเซ็น → OCR แม่นขึ้น
# ════════════════════════════════════════════════════════════════════
import cv2
import numpy as np


def has_table(img_path: str, min_area_ratio: float = 0.08) -> bool:
    """
    ตรวจสอบว่าภาพมีตารางหรือไม่
    min_area_ratio=0.08 → ตารางต้องใหญ่กว่า 8% ของภาพ
    """
    img = cv2.imread(img_path)
    if img is None: return True  # อ่านไม่ได้ → ลองส่ง OCR ดูก่อน

    h, w  = img.shape[:2]
    total = h * w

    gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )

    # Kernel ตรวจจับเส้นนอนและเส้นตั้ง
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 40, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 40))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)
    mask    = cv2.addWeighted(h_lines, 0.5, v_lines, 0.5, 0.0)
    _, mask = cv2.threshold(mask, 50, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return False

    max_area = max(cv2.contourArea(c) for c in contours)
    return (max_area / total) > min_area_ratio


def crop_table_region(img_path: str):
    """
    Crop เฉพาะส่วนตารางออกมา (ถ้าเจอ)
    คืน numpy array หรือ None ถ้าไม่เจอตาราง
    """
    img = cv2.imread(img_path)
    if img is None: return None

    h, w   = img.shape[:2]
    gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur   = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 40, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 40))
    h_lines  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_lines  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)
    mask     = cv2.addWeighted(h_lines, 0.5, v_lines, 0.5, 0.0)
    _, mask  = cv2.threshold(mask, 50, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None

    # หากล่องตารางที่ใหญ่ที่สุด
    best = max(contours, key=cv2.contourArea)
    if cv2.contourArea(best) < (h * w * 0.05): return None

    bx, by, bw, bh = cv2.boundingRect(best)
    pad = 20
    return img[max(0,by-pad):by+bh+pad, max(0,bx-pad):bx+bw+pad]


print('✅ Table filter functions ready')
print('  has_table()        → ตรวจว่าเป็นหน้าตาราง')
print('  crop_table_region() → crop เฉพาะส่วนตาราง')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 8  OCR Functions
#
#  2-Stage Pipeline:
#  Stage 1 (VLM): typhoon_ocr()    ภาพ → HTML table (structure prompt)
#  Stage 2 (LLM): extract_votes()  text → {party: votes} (Thai prompt)
#
#  Typhoon OCR Prompt: ต้องตรงตาม HuggingFace model card
#  mode='structure' → คืน HTML table → แม่นกว่าสำหรับ merged cells
# ════════════════════════════════════════════════════════════════════

def get_typhoon_prompt(mode: str = 'structure', anchor: str = '') -> str:
    """Typhoon OCR 3B prompt (ตาม HuggingFace model card)"""
    if mode == 'structure':
        return (
            'Below is an image of a document page, along with its dimensions and possibly '
            'some raw textual content previously extracted from it. '
            'Note that the text extraction may be incomplete or partially missing. '
            'Carefully consider both the layout and any available text to reconstruct the document accurately.\n'
            'Your task is to return the markdown representation of this document, '
            'presenting tables in HTML format as they naturally appear.\n'
            'If the document contains images or figures, analyze them and include the tag '
            '<figure>IMAGE_ANALYSIS</figure> in the appropriate location.\n'
            'Your final output must be in JSON format with a single key `natural_text` containing the response.\n'
            f'RAW_TEXT_START\n{anchor}\nRAW_TEXT_END'
        )
    return (
        'Below is an image of a document page along with its dimensions. '
        'Simply return the markdown representation of this document, '
        'presenting tables in markdown format as they naturally appear.\n'
        'If the document contains images, use a placeholder like dummy.png for each image.\n'
        'Your final output must be in JSON format with a single key `natural_text` containing the response.\n'
        f'RAW_TEXT_START\n{anchor}\nRAW_TEXT_END'
    )


# Extract Prompt (ภาษาไทย)
EXTRACT_PROMPT_TH = '''คุณคือผู้เชี่ยวชาญในการอ่านแบบฟอร์มผลการนับคะแนนเลือกตั้งไทย (สส.6/1)

งาน: จากข้อความ OCR ด้านล่าง ให้หาคะแนนของแต่ละพรรคในรายชื่อที่กำหนด

กฎ:
- คะแนนต้องเป็นตัวเลข Arabic (0-9) เท่านั้น
- ถ้าไม่พบพรรค ให้ใส่ 0
- ห้ามใส่คำอธิบาย ตอบเป็น JSON เท่านั้น
- รูปแบบ: {{"ชื่อพรรค": คะแนน, ...}}

รายชื่อพรรคที่ต้องหา:
{party_list}

ข้อความ OCR:
{ocr_text}'''.strip()


def typhoon_ocr(image_paths: List[str], doc_id: str) -> str:
    """
    Stage 1: ส่งภาพ → Typhoon OCR 3B (vLLM) → HTML table
    รองรับ multi-page (ส่งภาพทีละหน้า)
    """
    if not vllm_client: return ''
    all_pages = []

    for i, path in enumerate(image_paths):
        # ตรวจว่าเป็นหน้าตารางก่อน (ถ้าไม่ใช่ข้ามไป)
        if not has_table(path):
            print(f'    [Filter] {os.path.basename(path)} ข้ามเพราะไม่มีตาราง')
            continue

        cache_key = f'typhoon_{doc_id}_p{i+1}'
        cached    = load_cache(cache_key)
        if cached:
            all_pages.append(cached)
            continue

        b64    = img_to_b64(path)
        prompt = get_typhoon_prompt(mode='structure')

        for attempt in range(MAX_RETRIES):
            try:
                resp = vllm_client.chat.completions.create(
                    model       = 'typhoon-ocr',
                    messages    = [{'role': 'user', 'content': [
                        {'type': 'text', 'text': prompt},
                        {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64}'}}
                    ]}],
                    temperature = 0.0,
                    max_tokens  = 4096,
                )
                raw  = resp.choices[0].message.content or ''
                text = parse_typhoon_output(raw)
                # HTML Clean ทันที
                text = clean_ocr_output(text)
                save_cache(cache_key, text)
                all_pages.append(text)
                break
            except Exception as e:
                print(f'    [Typhoon OCR] p{i+1} attempt {attempt+1}: {e}')
                time.sleep(2 ** attempt)
        else:
            all_pages.append('')

    return '\n\n'.join(all_pages)


def extract_votes(ocr_text: str, party_list: List[str]) -> Dict[str, str]:
    """
    Stage 2: OCR text → {party: votes}
    ใช้ Typhoon LLM (text-only) กับ Thai prompt
    """
    if not vllm_client or not ocr_text: return {}

    prompt = EXTRACT_PROMPT_TH.format(
        party_list='\n'.join(f'- {p}' for p in party_list),
        ocr_text=ocr_text[:6000]
    )

    for attempt in range(MAX_RETRIES):
        try:
            resp = vllm_client.chat.completions.create(
                model       = 'typhoon-ocr',
                messages    = [{'role': 'user', 'content': prompt}],
                temperature = 0.0,
                max_tokens  = 2048,
            )
            result = parse_json(resp.choices[0].message.content or '')
            if result:
                return {k: normalize_vote(v) for k, v in result.items()}
        except Exception as e:
            print(f'    [Extract] attempt {attempt+1}: {e}')
            time.sleep(2 ** attempt)
    return {}


def zero_fix(party_name: str, ocr_text: str) -> str:
    """Reprompt เฉพาะพรรคที่ได้ 0 (focus เพื่อลด miss)"""
    if not vllm_client: return '0'
    prompt = (
        f'จากข้อความ OCR ด้านล่าง จงหาคะแนนของพรรค: {party_name}\n'
        f'ตอบเฉพาะตัวเลข Arabic เท่านั้น เช่น: 14813\n'
        f'ถ้าไม่พบ ตอบ: 0\n\n{ocr_text[:3000]}'
    )
    try:
        resp   = vllm_client.chat.completions.create(
            model='typhoon-ocr',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.0, max_tokens=32
        )
        digits = re.sub(r'[^\d]', '', (resp.choices[0].message.content or '').translate(_THAI))
        return digits.lstrip('0') or '0' if digits else '0'
    except:
        return '0'


import os
print('✅ OCR functions ready')
print('  typhoon_ocr()    → Stage 1: VLM (ภาพ → HTML table)')
print('  extract_votes()  → Stage 2: LLM (text → JSON votes, Thai prompt)')
print('  zero_fix()       → reprompt พรรคที่ได้ 0')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 9  Ingest Template CSV → doc_to_images map
# ════════════════════════════════════════════════════════════════════
import os, pandas as pd
from pathlib import Path
from collections import defaultdict


def ingest(template_path: str, image_dir: str):
    df = pd.read_csv(template_path)

    # รองรับทั้ง template ที่มี columns แล้วและไม่มี
    if 'party_name' not in df.columns and 'id' in df.columns:
        split = df['id'].str.rsplit('_', n=1, expand=True)
        df['doc_id']     = split[0]
        df['row_num']    = split[1].astype(int)
        df['party_name'] = df['id'].str.rsplit('_', n=1).str[0]  # fallback

    if 'doc_id' not in df.columns:
        df['doc_id']  = df['id'].str.rsplit('_', n=1).str[0]
        df['row_num'] = df['id'].str.rsplit('_', n=1).str[1].astype(int)

    if 'party_name' not in df.columns:
        # ใช้ชื่อคอลัมน์ที่ 2 (หลัง id) เป็น party_name
        cols = [c for c in df.columns if c not in ['id','doc_id','row_num','votes']]
        df['party_name'] = df[cols[0]] if cols else 'UNKNOWN'

    df['party_name'] = df['party_name'].fillna('UNKNOWN')

    # Map doc_id → image paths (รองรับ .png, .jpg, .jpeg + _page2, _page3 ...)
    doc_to_images = defaultdict(list)
    for doc_id in df['doc_id'].unique():
        for ext in ['png', 'jpg', 'jpeg']:
            p = Path(image_dir) / f'{doc_id}.{ext}'
            if p.exists():
                doc_to_images[doc_id].append(str(p))
                break
        for i in range(2, 10):
            for ext in ['png', 'jpg', 'jpeg']:
                pn = Path(image_dir) / f'{doc_id}_page{i}.{ext}'
                if pn.exists():
                    doc_to_images[doc_id].append(str(pn))
                    break

    found  = sum(1 for v in doc_to_images.values() if v)
    no_img = df['doc_id'].nunique() - found
    total  = sum(len(v) for v in doc_to_images.values())

    print('\n[ Ingest Summary ]')
    print(f'  Template rows : {len(df):,}')
    print(f'  Documents     : {df["doc_id"].nunique()}')
    print(f'  Images found  : {total} ({found} docs)')
    print(f'  No image      : {no_img} docs → submit 0')
    print(f'  Sample doc_id : {df["doc_id"].iloc[0]}')
    return df, dict(doc_to_images)


df_template, doc_to_images = ingest(TEMPLATE_PATH, DATA_DIR)


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 10  Main Pipeline
#
#  Flow สำหรับแต่ละ document:
#  ① typhoon_ocr(images) → OCR text (HTML → cleaned text)
#  ② extract_votes(text) → {party: votes}
#  ③ find_party_vote()   → fuzzy match
#  ④ zero_fix()          → reprompt ถ้าได้ 0
#  ⑤ Checkpoint ทุก 20 docs
# ════════════════════════════════════════════════════════════════════

def process_doc(doc_id: str, doc_rows, image_paths) -> Dict[str, str]:
    results    = {r['id']: '0' for _, r in doc_rows.iterrows()}
    party_list = doc_rows.sort_values('row_num')['party_name'].tolist() \
                 if 'row_num' in doc_rows.columns \
                 else doc_rows['party_name'].tolist()

    if not image_paths:
        return results

    # Stage 1: OCR (includes table filter + HTML clean)
    ocr_text = typhoon_ocr(image_paths, doc_id)
    if not ocr_text:
        return results

    # Stage 2: Extract votes
    votes = extract_votes(ocr_text, party_list)

    # Map to submission rows
    for _, row in doc_rows.iterrows():
        row_id = row['id']
        party  = row['party_name']
        vote   = find_party_vote(party, votes)

        # Zero Fix
        if vote == '0' and ocr_text:
            vote = zero_fix(party, ocr_text)

        results[row_id] = vote

    return results


def run_pipeline():
    print('=' * 65)
    print('  Thai Election OCR Pipeline — Typhoon OCR 3B Local')
    print(f'  Environment : {ENV_NAME}')
    print(f'  Model       : {TYPHOON_MODEL}')
    print(f'  GPU         : {GPU_TIER} ({vram_gb:.0f}GB)')
    print(f'  Cache       : {USE_CACHE} ({CACHE_DIR})')
    print('=' * 65)

    all_results = {}
    grouped     = df_template.groupby('doc_id')
    doc_ids     = sorted(grouped.groups.keys())

    for idx, doc_id in enumerate(tqdm(doc_ids, desc='Pipeline')):
        doc_rows    = grouped.get_group(doc_id).reset_index(drop=True)
        image_paths = doc_to_images.get(doc_id, [])
        res         = process_doc(doc_id, doc_rows, image_paths)
        all_results.update(res)

        # ── Checkpoint ทุก 20 docs ──────────────────────────────
        if (idx + 1) % 20 == 0:
            df_ck = pd.DataFrame([
                {'id': r['id'], 'votes': all_results.get(r['id'], '0')}
                for _, r in df_template.iterrows()
            ])
            ck_path = f'{BASE_DIR}/submission_checkpoint.csv'
            df_ck.to_csv(ck_path, index=False)
            filled = (df_ck['votes'] != '0').sum()
            print(f'  💾 Checkpoint [{idx+1}/{len(doc_ids)}]: {filled:,} rows filled → {ck_path}')

    # Final submission
    df_out = pd.DataFrame([
        {'id': r['id'], 'votes': all_results.get(r['id'], '0')}
        for _, r in df_template.iterrows()
    ])
    df_out.to_csv(OUTPUT_PATH, index=False)

    zeros   = (df_out['votes'] == '0').sum()
    nonzero = len(df_out) - zeros
    print(f'\n✅ Saved → {OUTPUT_PATH}')
    print(f'   Total rows  : {len(df_out):,}')
    print(f'   Non-zero    : {nonzero:,} ({nonzero/len(df_out)*100:.1f}%)')
    print(f'   Zeros       : {zeros:,} ({zeros/len(df_out)*100:.1f}%)')
    return df_out


df_submission = run_pipeline()


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 11  Final Check + Download submission.csv
# ════════════════════════════════════════════════════════════════════

def final_check():
    df_tpl = pd.read_csv(TEMPLATE_PATH)
    df_sub = pd.read_csv(OUTPUT_PATH)

    # Normalize column names
    if 'id' not in df_tpl.columns:
        df_tpl.columns = ['id'] + list(df_tpl.columns[1:])

    missing = set(df_tpl['id']) - set(df_sub['id'])
    print('[ Final Check ]')
    print(f'  Rows    : {len(df_sub):,} / {len(df_tpl):,}')
    print(f'  Missing : {len(missing)}')

    # แก้ค่าที่ไม่ใช่ตัวเลข → 0
    df_sub['votes'] = df_sub['votes'].astype(str)
    bad = ~df_sub['votes'].str.match(r'^\d+$')
    if bad.sum():
        df_sub.loc[bad, 'votes'] = '0'
        print(f'  Fixed   : {bad.sum()} non-digit → 0')

    zeros = (df_sub['votes'] == '0').sum()
    print(f'  Zeros   : {zeros:,} ({zeros/len(df_sub)*100:.1f}%)')

    df_sub[['id','votes']].to_csv(OUTPUT_PATH, index=False)
    ok = len(missing) == 0
    print(f'  Status  : {"✅ VALID" if ok else "❌ ERROR — มี missing rows"}')
    return ok


is_valid = final_check()
print()
print(pd.read_csv(OUTPUT_PATH).head(10).to_string(index=False))

if is_valid:
    if IS_COLAB:
        from google.colab import files
        files.download(OUTPUT_PATH)
        print(f'\n✅ Download started')
    else:
        print(f'\n✅ ไฟล์พร้อม: {OUTPUT_PATH}')
        print('   RunPod: ดาวน์โหลดผ่าน File Manager ใน JupyterLab')
        print('   หรือใช้: scp pod@<ip>:{OUTPUT_PATH} ./')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  CELL 12  Debug: ทดสอบ OCR + HTML Cleaner กับ 1 ภาพ
# ════════════════════════════════════════════════════════════════════
import json

TEST_DOC   = 'constituency_10_1'  # แก้ตรงนี้เพื่อทดสอบ doc อื่น
test_paths = doc_to_images.get(TEST_DOC, [])
print(f'Testing : {TEST_DOC}')
print(f'Images  : {test_paths}')

if test_paths:
    print(f'\n▶ Table filter check:')
    for p in test_paths:
        has_t = has_table(p)
        print(f'  {os.path.basename(p)}: {"✅ มีตาราง" if has_t else "❌ ไม่มีตาราง (จะข้าม)"}')

if test_paths and vllm_client:
    print('\n▶ Typhoon OCR (หน้าแรก):')
    raw_ocr = typhoon_ocr(test_paths[:1], TEST_DOC + '_debug')
    print(raw_ocr[:2000])

    print('\n▶ Extract votes:')
    rows    = df_template[df_template['doc_id'] == TEST_DOC]
    parties = rows['party_name'].tolist()
    votes   = extract_votes(raw_ocr, parties)
    nonzero = {k: v for k, v in votes.items() if v != '0'}
    print(f'  Found {len(nonzero)} non-zero parties:')
    print(json.dumps(nonzero, ensure_ascii=False, indent=2)[:1000])
else:
    print('vLLM ไม่พร้อม หรือไม่พบภาพ')


# 🖥️ RunPod Setup Guide

## วิธีใช้กับ RunPod

### 1. สร้าง Pod
```
Template   : PyTorch 2.x (runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04)
GPU        : A100 SXM (40GB) หรือ H100 NVL (94GB)
Disk       : 50GB Container + 50GB Volume
Expose     : Port 8888 (Jupyter) + Port 8000 (vLLM)
```

### 2. เปิด JupyterLab
```
https://<pod-id>-8888.proxy.runpod.net/lab
```

### 3. Upload Files
```bash
# Upload ผ่าน JupyterLab File Manager:
# - kaggle.json → /workspace/kaggle.json
# - notebook นี้ → /workspace/
```

### 4. รัน Cells ตามลำดับ 1 → 12

### 5. Download submission.csv
```bash
# วิธี 1: JupyterLab → คลิกขวา submission.csv → Download
# วิธี 2: Terminal บน local
scp root@<pod-ip>:/workspace/submission.csv ./

# วิธี 3: runpodctl
runpodctl receive <pod-id> /workspace/submission.csv
```

### 6. หยุด Pod หลังใช้งาน
```
สำคัญมาก! หยุด Pod ใน RunPod dashboard เพื่อหยุดค่าใช้จ่าย
```

---

## Cost Estimate
| GPU | ราคา/ชม | เวลารัน | ราคาประมาณ |
|---|---|---|---|
| A100 SXM 40GB | ~$1.89/hr | ~3-4 ชม | ~$7 |
| H100 NVL | ~$3.69/hr | ~2-3 ชม | ~$9 |
| RTX 4090 24GB | ~$0.69/hr | ~5-6 ชม | ~$4 |
